# Datamine DAELLIPS Process Example

This notebook demonstrates how to configure and run the **`daellips`** process wrapper in `dmstudio`.

## Process Description

## DAELLIPS

Create ellipsoids from dynamic anisotropy points or model.

Input is either a points or a block model file (not both).

Typically, points data will be created by the **ANISOANG** process, although it can be from other sources. An input points file must contain coordinate fields (**XPT** /**YPT** /**ZPT**) and at least one angle field. Up to three angle fields can be specified, representing the dip direction (**DIPDIR**), dip (**DIP**) and roll (**ROLL**). Providing at least one directional field is set, the remaining orientations can be static and specified using the @**DIPDIR** , @**DIP** and/or @**ROLL** field.

[![](../Images/DAELLIPS.png)](<javascript:void\(0\);>)

Model data will typically be as produced by anisotropic modelling functions. As with points, each cell record must contain at least one angle field.

Ellipsoid centroids, derived from either &**POINTS** or &**MODEL** can be optionally declustered (multiple options) and 3 radii are specified to determine the final ellipsoid shape. @**XGRID** , @**YGRID** and @**ZGRID** parameters will define the declustering grid (if declustering is applied). The original of the grid can be specified (@**XORIG** , @**YORIG** and @**ZORIG**) if the input data is of the points type.

You can also choose (@**CENTRE**) as to how the ellipsoid centroid is positioned, either at the point or subcell centroid (0) or on the centre of the grid or the parent cell within which the point lies.

In all cases, output will be a data file of the ellipsoids data type, and can be loaded into a 3D view to visualize ellipsoid location(s) and orientation(s).

### Input Files:

* **points** (Points):
  Angle data in points format created by ANISOANG. It must include XPT, YPT, ZPT and at
  least one angle field. Either a POINTS or a MODEL file must be selected but not both.
  Required=No

* **model** (Block Model):
  Angle data in model format created during dynamic anisotropy modelling. It must include
  at least one angle field. Either a POINTS or a MODEL file must be selected but not both.
  Required=No

### Output Files:

* **ellipses** (Ellipsoid):
  Output ellipsoids. Ellipsoids to be displayed in the 3D Window.
  Required=Yes

### Fields:

* **angles** (Undefined : Undefined):
  A field in the POINTS or MODEL file that includes rotation around AXIS1. AXIS1 specified in Parameters.
  Default=Undefined
  Required=No

* **zone** (Undefined : POINTS or MODEL):
  Optional attribute field (numeric or alphanumeric) in input points or model. This will
  create an output ellipse for each zone in the input file, and ZONE will be assigned to
  output ellipsoid.
  Default=Undefined
  Required=No

### Parameters:

* **declust**:
  Flag to select declustering option. Default (0). **0** :do not use declustering. **1**
  :random selection within each grid cell (different selection for each run). **2** :
  pseudo random selection within each grid cell (repeatable).. **3** : nearest to grid
  centre.
  Range=0,3
  Values=0,1,2,3
  Default=Yes
  Required=No

* **factor**:
  Dividing factor for adjusting size of RAD1, RAD2 and RAD3.
  Range=1
  Values=1
  Default=1
  Required=Yes

* **axis1**:
  AXIS1 for rotation where 1=X, 2=Y and 3=Z. Default (3).
  Range=1,3
  Values=1,2,3
  Default=3
  Required=No

* **axis2**:
  AXIS2 for rotation where 1=X, 2=Y and 3=Z. Default (1).
  Range=1,3
  Values=1,2,3
  Default=1
  Required=No

* **axis3**:
  AXIS3 for rotation where 1=X, 2=Y and 3=Z. Default (3).
  Range=1,3
  Values=1,2,3
  Default=3
  Required=No

* **centre**:
  Flag to define where the ellipsoid should be located. Only required if @DECLUST > 0.
  **0** : centre the ellipsoid on the coordinates of the point or subcell centre. **1** :
  centre the ellipsoid on the centre of the grid or parent cell within which the point
  lies. .
  Range=0,1
  Values=0,1
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('daellips')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute daellips
print("Running daellips...")
dm_cmd.daellips(
    model_i='t_mod1',  # required
    ellipses_o=['t_daellips_out'],  # required
    declust_p='Yes',  # required
    factor_p='required_val',  # required
    # points_i=['t_SurfacePointsPt'],  # optional
    # angles_f=['optional'],  # optional
    # zone_f='optional',  # optional
    # axiss_f=['optional'],  # optional
    # centre_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("daellips execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_daellips_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")